# QLoRA fine-tune of the Spec RAG drafter (companion notebook)

This is the paper's drafter recipe: fine-tune a 7B model to produce *(rationale, ranking)* from *(user history, plot evidence, candidates)*. The rationales were written by Gemini in §10 of the main notebook, from users and a time window that the test evaluation never uses.

**Run in a fresh session** (GPU T4 ×2, Internet on). Unsloth and vLLM pin different torch builds, so they don't share an environment. Training uses one T4.

1. Attach the dataset `specrag-teacher`, containing `teacher_train.jsonl` from the main notebook.
2. Run all cells. Then save `/kaggle/working/drafter_lora` as the dataset `specrag-drafter-lora` and attach it to the main notebook for §10b.

Expected memory use: 7B in 4-bit ≈ 5 GB, plus LoRA r=16 and gradient checkpointing at 4k tokens ≈ 10–12 GB on one T4.

In [ ]:
!pip install -q unsloth 2>&1 | tail -n 3

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import json, inspect
from pathlib import Path

from unsloth import FastLanguageModel
from unsloth.chat_templates import train_on_responses_only
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

SEED = 42
DATA = Path("/kaggle/input/specrag-teacher/teacher_train.jsonl")
OUT = Path("/kaggle/working/drafter_lora")
BASE = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"   # same weights as the AWQ model the main notebook serves
MAX_LEN = 4096

model, tok = FastLanguageModel.from_pretrained(BASE, max_seq_length=MAX_LEN, load_in_4bit=True, dtype=None)
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=16, lora_dropout=0, bias="none", random_state=SEED,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth")

In [ ]:
ds = load_dataset("json", data_files=str(DATA), split="train")
ds = ds.map(lambda ex: {"text": tok.apply_chat_template(ex["messages"], tokenize=False)}, remove_columns=["messages"])
n_tok = [len(tok(t)["input_ids"]) for t in ds.select(range(min(200, len(ds))))["text"]]
print(len(ds), "examples | tokens p50/max on a sample:", sorted(n_tok)[len(n_tok) // 2], max(n_tok))
split = ds.train_test_split(test_size=0.05, seed=SEED)

# TRL renamed max_seq_length -> max_length; pass whichever this version accepts
len_kw = "max_length" if "max_length" in inspect.signature(SFTConfig).parameters else "max_seq_length"
args = SFTConfig(
    output_dir="/kaggle/working/qlora_out", dataset_text_field="text", **{len_kw: MAX_LEN},
    per_device_train_batch_size=1, gradient_accumulation_steps=8, num_train_epochs=1,
    learning_rate=2e-4, lr_scheduler_type="cosine", warmup_ratio=0.03, weight_decay=0.0,
    fp16=True, bf16=False, optim="adamw_8bit", logging_steps=10,
    eval_strategy="steps", eval_steps=50, per_device_eval_batch_size=1,
    save_strategy="no", seed=SEED, report_to="none")
trainer = SFTTrainer(model=model, processing_class=tok, train_dataset=split["train"], eval_dataset=split["test"], args=args)
# loss only on the assistant turn (rationale + ranking), not on the prompt
trainer = train_on_responses_only(trainer, instruction_part="<|im_start|>user\n", response_part="<|im_start|>assistant\n")
stats = trainer.train()
print(stats)

In [ ]:
model.save_pretrained(OUT)
tok.save_pretrained(OUT)
print(sorted(p.name for p in OUT.iterdir()))

**Sanity check before saving the dataset.** Generate one held-out example and check that the output is valid JSON with a rationale followed by a ranking. If it isn't, the eval loss above should show what went wrong.

In [ ]:
FastLanguageModel.for_inference(model)
ex = split["test"][0]["text"]
prompt = ex[: ex.rindex("<|im_start|>assistant\n") + len("<|im_start|>assistant\n")]
ids = tok(prompt, return_tensors="pt", add_special_tokens=False).to("cuda")
out = model.generate(**ids, max_new_tokens=500, do_sample=False)
print(tok.decode(out[0, ids["input_ids"].shape[1]:], skip_special_tokens=True))